In [271]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
# !pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [272]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <stdint.h> // uint32_t

using element_t = uint32_t;

In [273]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [274]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <random>  // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {

/**
 * Generates and returns a vector of random uniform data of a given length, n,
 * for any integral type. Input range will be [0, 2n].
 */
template < typename T >
std::vector< T > generate_uniform( std::size_t n )
{
    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20240916;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 2 * n);

    std::vector< T > random_data( n ); // init array
    std::generate( std::begin( random_data )
                 , std::end  ( random_data )
                 , [ &rng, &distrib ](){ return static_cast< T >( distrib( rng ) ); });

    return random_data;
}


} // namespace a1
} // namespace csc485b

In [275]:
%%cuda_group_save -g "source" -n "algorithm_choices.h"
#pragma once

#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {
namespace cpu {

void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace cpu


namespace gpu {

void run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [276]:
%%cuda_group_save -g "source" -n "cpu_baseline.cu"
/**
 * CPU methods that the GPU should outperform.
 */

#include "algorithm_choices.h"

#include <algorithm> // std::sort()
#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

namespace csc485b {
namespace a1      {
namespace cpu     {

/**
 * Simple solution that just sorts the whole array with a built-in sort
 * function and then resorts the last portion in the opposing order with
 * a second call to that same built-in sort function.
 */
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    std::sort( data, data + num_elements, std::less< element_t >{} );
    std::sort( data + invert_at_pos, data + num_elements, std::greater< element_t >{} );
}

/**
 * Run the single-threaded CPU baseline that students are supposed to outperform
 * in order to obtain higher grades on this assignment. Times the execution and
 * prints to the standard output (e.g., the screen) that "wall time." Note that
 * the functions takes the input by value so as to not perturb the original data
 * in place.
 */
void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    auto const cpu_start = std::chrono::high_resolution_clock::now();
    opposing_sort( data.data(), switch_at, n );
    auto const cpu_end = std::chrono::high_resolution_clock::now();

    std::cout << "CPU Baseline time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(cpu_end - cpu_start).count()
              << " ns" << std::endl;

    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace cpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "gpu_solution.cu"
/**
 * The file in which you will implement your GPU solutions!
 */

#include "algorithm_choices.h"

#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

#include <algorithm>

#include "cuda_common.h"

namespace{
    int const MAX_WARPS = 32;
    int const MAX_LANES = 32;
}

namespace csc485b {
namespace a1      {
namespace gpu     {

/**
 * The CPU baseline benefits from warm caches because the data was generated on
 * the CPU. Run the data through the GPU once with some arbitrary logic to
 * ensure that the GPU cache is warm too and the comparison is more fair.
 */
__global__
void warm_the_gpu( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    // We know this will never be true, because of the data generator logic,
    // but I doubt that the compiler will figure it out. Thus every element
    // should be read, but none of them should be modified.
    if( th_id < num_elements && data[ th_id ] > num_elements * 100 )
    {
        ++data[ th_id ]; // should not be possible.
    }
}


/**
 * Your solution. Should match the CPU output.
 */
__global__
void bitonic_sort( element_t * data, std::size_t num_elements, int initial_stride, int sub_stride, int threads_per_block)
{
    //init shared memory
     __shared__ element_t smem [MAX_WARPS][MAX_LANES];

    int warp_id = threadIdx.y;
    int lane_id = threadIdx.x;
    int const th_id = (blockIdx.x * (blockDim.x * blockDim.y) + ((warp_id * blockDim.x) + lane_id));

    //copy from global to shared
    smem[warp_id][lane_id] = data[th_id];
    __syncthreads();
    
    //calculate direction of the swap
    int global_group_size = initial_stride * 2;
    int group_number = th_id/global_group_size;
    int dir = (group_number % 2 == 0); //if 1 then ascending, 0 then descending

    int sub_group_size = sub_stride * 2;
    int head = (th_id / sub_group_size) * sub_group_size; //integer division

    //only swap if (head <= th_id < half of subgroup)
    if ((th_id < num_elements) && (th_id + sub_stride < num_elements) && 
        (th_id < head + (sub_group_size/2)) && (th_id + sub_stride < head + sub_group_size)) {
    
        // if the substride is GREATER THAN the threads available in a block, then use global memory
        if (sub_stride >= threads_per_block) {
            int target_id = th_id + sub_stride;

            if ((data[th_id] < data[target_id]) != dir) {
                element_t temp = data[th_id];
                data[th_id] = data[target_id];
                data[target_id] = temp;
            }
        }

        // if the substride is LESS THAN the threads available in a block, then use shared memory
        else {
            // wrap around if (lane_id + stride) > 32
            int target_warp = warp_id;
            int target_lane = lane_id + sub_stride;
            target_warp += target_lane/32;
            target_lane = target_lane%32;

            if ((smem[warp_id][lane_id] < smem[target_warp][target_lane]) != dir) {
                element_t temp = smem[warp_id][lane_id];
                smem[warp_id][lane_id] = smem[target_warp][target_lane];
                smem[target_warp][target_lane] = temp;

                __syncthreads();
                data[th_id] = smem[warp_id][lane_id];
                data[th_id + sub_stride] = smem[target_warp][target_lane];
            }
        }
    }
    
    return;
}

__global__
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements, int kernel_launch_index, int threads_per_block, int num_blocks){
    // I dont need shared memory because all operations are atomic and independent

    int warp_id = threadIdx.y;
    int lane_id = threadIdx.x;
    int const th_id = (blockIdx.x * (blockDim.x * blockDim.y) + ((warp_id * blockDim.x) + lane_id));

    int source_data_index = invert_at_pos + (kernel_launch_index * threads_per_block * num_blocks) + th_id;
    int target_data_index = num_elements - 1 - (kernel_launch_index * threads_per_block * num_blocks) - th_id;

    if (source_data_index >= (invert_at_pos + num_elements)/2 || source_data_index >= num_elements-1){return;}

    int temp = data[source_data_index];
    data[source_data_index] = data[target_data_index];
    data[target_data_index] = temp;

    // printf("source = %d\n", temp);
}

/**
 * Performs all the logic of allocating device vectors and copying host/input
 * vectors to the device. Times the opposing_sort() kernel with wall time,
 * but excludes set up and tear down costs such as mallocs, frees, and memcpies.
 */
void run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    // Kernel launch configurations. Feel free to change these.
    // This is set to maximise the size of a thread block on a T4, but it hasn't
    // been tuned. It's not known if this is optimal.
    dim3 threads(MAX_LANES, MAX_WARPS);
    std::size_t const threads_per_block = threads.x * threads.y;
    std::size_t const num_blocks =  ( n + threads_per_block - 1 ) / threads_per_block;

    // Allocate arrays on the device/GPU
    element_t * d_data;
    cudaMalloc( (void**) & d_data, sizeof( element_t ) * n );
    CHECK_ERROR("Allocating input array on device");

    // Copy the input from the host to the device/GPU
    cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
    CHECK_ERROR("Copying input array to device");

    // Warm the cache on the GPU for a more fair comparison
    warm_the_gpu<<< num_blocks, threads>>>( d_data, switch_at, n );

    // Time the execution of the kernel that you implemented
    auto const kernel_start = std::chrono::high_resolution_clock::now();

    // start host loop
    for(int initial_stride = 1; initial_stride <= n/2; initial_stride <<= 1){
        for (int sub_stride = initial_stride; sub_stride>0; sub_stride >>= 1){
            bitonic_sort<<< num_blocks, threads>>>( d_data, n , initial_stride, sub_stride, threads_per_block);
            
        }
    }

    cudaDeviceSynchronize();

    // start host loop
    //TODO: I want to loop throuhg the last 1/4 elements and reverse the ordering. so index 0 will become index n-1, etc. runtime is O((n/8)
    // How to do this parallelly. I want to have each thread in charge of swapping one element. if there are more data than threads, they will double up their tasks in a loop

    int threads_per_launch = threads_per_block * num_blocks;
    int total_loops = ((n / 8) + threads_per_launch - 1) / threads_per_launch;

    for (int kernel_launch_index=0; kernel_launch_index< total_loops; kernel_launch_index++){
        opposing_sort<<<num_blocks, threads>>>( d_data, switch_at, n, kernel_launch_index, threads_per_block, num_blocks);
    }

    cudaDeviceSynchronize();

    auto const kernel_end = std::chrono::high_resolution_clock::now();
    CHECK_ERROR("Executing kernel on device");

    // After the timer ends, copy the result back, free the device vector,
    // and echo out the timings and the results.
    cudaMemcpy( data.data(), d_data, sizeof( element_t ) * n, cudaMemcpyDeviceToHost );
    CHECK_ERROR("Transferring result back to host");
    cudaFree( d_data );
    CHECK_ERROR("Freeing device memory");

    std::cout << "GPU Solution time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(kernel_end - kernel_start).count()
              << " ns" << std::endl;

    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [278]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <cstddef>  // std::size_t type
#include <iostream> // std::cout, std::endl
#include <vector>

#include "algorithm_choices.h"
#include "data_generator.h"
#include "data_types.h"
#include "cuda_common.h"
#include <cmath>

int main()
{
    // this part is 3 * n/4
    std::size_t const n = pow(2,12);
    std::size_t const switch_at = 3 * ( n >> 2 ) ;

    auto data = csc485b::a1::generate_uniform< element_t >( n );

    //print start data values
    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;

    csc485b::a1::cpu::run_cpu_baseline( data, switch_at, n );
    csc485b::a1::gpu::run_gpu_soln( data, switch_at, n );

    return EXIT_SUCCESS;
}

In [279]:
%cuda_group_run --group "source" --compiler-args "-O0 -g -std=c++20 -arch=sm_75"

162 2914 1654 4982 6582 6673 7551 27 1364 6527 1403 849 4015 4463 4642 1714 1683 7111 4352 54 2486 1064 214 3488 4193 946 1304 2433 3459 1263 7223 1921 3454 2718 5346 4289 335 4105 7944 2919 1182 3711 1954 1222 1858 3889 2305 4256 4881 490 4907 5314 1139 196 2969 4584 3045 5821 1886 808 3053 6462 4748 4207 6336 2845 5811 3993 5987 5160 3969 6072 7811 2466 6024 7991 3727 4910 3323 6462 3688 7076 6609 8179 2239 7818 3159 5776 7092 2730 7937 2718 4289 2772 955 6653 7783 6342 2299 1872 1096 4547 6077 5972 284 3119 6152 849 4041 1919 1336 5324 526 7826 3018 799 1721 5633 1884 6738 5190 565 6003 2875 6132 5544 3069 5965 2365 2056 2301 2755 1016 6601 4688 2779 3239 1317 5836 601 2615 1933 6138 8104 443 5697 5511 1261 404 2222 3262 7696 286 5313 1942 3768 5859 5292 4994 4436 5203 1105 2972 5133 3073 1496 887 6625 954 6540 5369 6721 6290 6898 6045 3755 3376 3525 2381 1770 5636 8116 6233 6343 2034 8011 3822 6202 5652 2794 719 5000 6089 3346 8000 2398 3701 3926 8159 7172 7634 7974 7066 7128 5867 